In [ ]:
# revT2V — Kaggle dataset builder. Generates NEW latent shards from prompts/train.txt
# via scripts/build_dataset.py and pushes them to a separate Hub folder, so the
# existing 154 root shards in HF_DATASET_REPO are never touched or overwritten.
# Needs: GPU on, Internet on, Secret HF_TOKEN attached (write access to HF_DATASET_REPO).
# Settings
HF_DATASET_REPO = "silentlooop/revt2v-data"  # already has 154 shards at the repo root ("existing")
NEW_DATA_VERSION = "new"  # Hub folder the freshly-built shards go to ("new"); root stays untouched
REPO_URL = "https://github.com/silentlooop/revT2V.git"
REPO_DIR = "/kaggle/working/revT2V"

CONFIG = "configs/conv_lora.yaml"  # only its shared video/model settings are used (not the method)
PROMPTS_FILE = "prompts/train.txt"
TEST_PROMPTS_FILE = "prompts/test.txt"  # only checked for train/test overlap, never built
SEEDS_PER_PROMPT = 3
MIN_MOTION = 0.15
PUSH_EVERY = 10  # clips per Hub commit
LIMIT = None  # set e.g. 5 to smoke-test the pipeline before committing to a full run

print(f"Building from {PROMPTS_FILE} -> {HF_DATASET_REPO}/{NEW_DATA_VERSION}/ (existing data stays at the repo root)")


In [ ]:
# 1. Clone/pull the repo and install it
!git clone -q {REPO_URL} {REPO_DIR} 2>/dev/null || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -e ".[eval]"
# Kaggle preinstalls torchao 0.10.0, which peft rejects outright (it requires >=0.16.0
# for its optional torchao LoRA dispatcher) — we never use torchao-quantized layers,
# so removing it avoids the crash without touching torch/CUDA-adjacent packages.
!pip uninstall -y torchao -q


In [ ]:
# 2. HF token from Kaggle Secrets (needs write access to HF_DATASET_REPO)
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Kaggle Secret HF_TOKEN is empty / not attached to this notebook"

from huggingface_hub import whoami
print("HF user:", whoami(token=os.environ["HF_TOKEN"])["name"])


In [ ]:
# 3. GPU check (the teacher pipeline needs one to generate clips at a usable speed)
import torch

print("torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU visible — Settings > Accelerator > GPU"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")


In [ ]:
# 4. Helper
import subprocess
import sys


def run(*args):
    """Run a command, stream its output live, fail the notebook on a non-zero exit, return the output."""
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return "".join(lines)


In [ ]:
# 5. Build the new shards and push them to HF_DATASET_REPO/NEW_DATA_VERSION/
# Local output goes to its own directory (not data/latents) so it never collides
# with any existing shards pulled in as a Kaggle input for other notebooks.
OUTPUT_DIR = f"data/{NEW_DATA_VERSION}"

build_args = [
    sys.executable, "scripts/build_dataset.py",
    "--config", CONFIG,
    "--prompts", PROMPTS_FILE,
    "--test-prompts", TEST_PROMPTS_FILE,
    "--output", OUTPUT_DIR,
    "--hf-repo", HF_DATASET_REPO,
    "--data-version", NEW_DATA_VERSION,
    "--seeds-per-prompt", SEEDS_PER_PROMPT,
    "--min-motion", MIN_MOTION,
    "--push-every", PUSH_EVERY,
]
if LIMIT:
    build_args += ["--limit", LIMIT]

out = run(*build_args)
open("/kaggle/working/build_dataset.log", "w").write(out)


In [ ]:
# 6. Verify the Hub split: existing shards at the repo root, new ones under NEW_DATA_VERSION/
from revt2v.utils import list_hub_files

hub_files = list_hub_files(HF_DATASET_REPO, repo_type="dataset")
existing_shards = [f for f in hub_files if f.endswith(".pt") and "/" not in f]
new_shards = [f for f in hub_files if f.endswith(".pt") and f.startswith(f"{NEW_DATA_VERSION}/")]

print(f"existing (repo root): {len(existing_shards)} shards — untouched")
print(f"new      ({NEW_DATA_VERSION}/):   {len(new_shards)} shards")
assert f"{NEW_DATA_VERSION}/manifest.jsonl" in hub_files, "new manifest didn't make it to the Hub"
print(f"manifest pushed: {HF_DATASET_REPO}/{NEW_DATA_VERSION}/manifest.jsonl")


In [ ]:
# 7. Outputs (shards + manifest are also on the HF Hub, under NEW_DATA_VERSION/)
!ls -la /kaggle/working
!find {OUTPUT_DIR} -maxdepth 1 -name "*.pt" | wc -l
